《Think Stats》 3판은 [Bookshop.org](https://bookshop.org/a/98697/9781098190255)와 [Amazon](https://amzn.to/42lmxwu)에서 구할 수 있습니다. 두 링크는 원저자의 제휴 링크입니다. 책이 도움이 되었다면 Green Tea Press의 [무료 공개 교육 자료 제작을 후원](https://greenteapress.com/support/?book=ts3)하는 것도 생각해 주세요.

# 분포 모형

지금까지는 관측 자료로 만든 경험분포를 사용했습니다. 이런 분포 가운데 일부는 간단한 수학 함수로 나타내는 이론분포로 근사할 수 있습니다. 이 장에서는 이런 이론분포 몇 가지와, 그 분포로 설명할 수 있는 자료를 소개합니다.

다음과 같은 예를 살펴보겠습니다.

- 스키트 사격 경기의 명중 횟수와 실패 횟수는 이항분포로 잘 설명됩니다.
- 아이스하키나 축구의 득점 수는 포아송분포로, 득점 사이의 시간은 지수분포로 근사할 수 있습니다.
- 출생 체중은 정규분포로, 성인의 체중은 로그정규분포로 어느 정도 설명할 수 있습니다. 정규분포는 가우스분포라고도 합니다.

이 분포나 운동 경기에 익숙하지 않아도 필요한 내용을 설명하겠습니다. 각 예제에서는 간단한 모형에 따른 시뮬레이션부터 시작해, 결과가 이론분포를 따르는지 확인합니다. 그런 다음 실제 자료가 모형과 얼마나 잘 맞는지 살펴보겠습니다.

[이 노트북의 원문을 Colab에서 실행하기](https://colab.research.google.com/github/AllenDowney/ThinkStats/blob/v3/nb/chap05.ipynb).

In [1]:
from os.path import basename, exists


def download(url):
    filename = basename(url)
    if not exists(filename):
        from urllib.request import urlretrieve

        local, _ = urlretrieve(url, filename)
        print("Downloaded " + local)


download("https://github.com/AllenDowney/ThinkStats/raw/v3/nb/thinkstats.py")

In [2]:
try:
    import empiricaldist
except ImportError:
    %pip install empiricaldist

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from thinkstats import decorate

## 이항분포

첫 번째 예는 공중으로 날아가는 점토 표적을 산탄총으로 맞히는 스키트 사격입니다. 원문에서 다루는 올림픽을 포함한 국제 경기에서는 라운드당 표적 25개씩 다섯 라운드를 치르며, 승자를 가리기 위해 필요하면 추가 라운드를 진행합니다.

이 경기를 설명하는 모형으로, 모든 참가자의 명중 확률이 매번 `p`로 같고, 각 시도의 결과는 서로 독립이라고 가정하겠습니다. 물론 단순화한 모형입니다. 실제로는 선수마다 명중 확률이 다르고, 같은 선수도 시도에 따라 달라질 수 있습니다. 이런 단순한 모형이 실제 결과를 얼마나 잘 설명하는지 확인해 보겠습니다.

다음 함수로 모형을 시뮬레이션하겠습니다. 표적 수 `n`과 각 표적의 명중 확률 `p`를 받아, 명중과 실패를 나타내는 1과 0의 시퀀스를 반환합니다.

In [4]:
def flip(n, p):
    choices = [1, 0]
    probs = [p, 1 - p]
    return np.random.choice(choices, n, p=probs)

각 표적의 명중 확률이 90%일 때 표적 25개로 된 한 라운드를 시뮬레이션하는 예입니다.

In [5]:
# Seed the random number generator so we get the same results every time
np.random.seed(1)

In [6]:
flip(25, 0.9)

더 긴 시퀀스를 생성하고 결과의 `Pmf`를 구하면, 1과 0의 비율이 대략 예상대로인지 확인할 수 있습니다.

In [7]:
from empiricaldist import Pmf

seq = flip(1000, 0.9)
pmf = Pmf.from_seq(seq)
pmf

이제 `flip`으로 스키트 사격 한 라운드를 시뮬레이션하고 명중 횟수를 반환할 수 있습니다.

In [8]:
def simulate_round(n, p):
    seq = flip(n, p)
    return seq.sum()

큰 대회에서 선수 200명이 다섯 라운드씩 사격하고, 모두의 명중 확률이 `p=0.9`라고 하겠습니다. `simulate_round`를 1,000번 호출하면 이런 대회를 시뮬레이션할 수 있습니다.

In [9]:
n = 25
p = 0.9
results_sim = [simulate_round(n, p) for i in range(1000)]

평균 점수는 `n`과 `p`의 곱인 `22.5`에 가깝습니다.

In [10]:
np.mean(results_sim), n * p

결과의 분포는 다음과 같습니다.

In [11]:
from empiricaldist import Pmf

pmf_sim = Pmf.from_seq(results_sim, name="simulation results")

pmf_sim.bar()
decorate(xlabel="Hits", ylabel="PMF")

봉우리는 평균 근처에 있고, 분포는 왼쪽으로 치우쳐 있습니다.

시뮬레이션을 하지 않고 이 분포를 예측할 수도 있습니다. 수학적으로 이 결과는 **이항분포**(binomial distribution)를 따르며, 그 PMF는 쉽게 계산할 수 있습니다.

In [12]:
from scipy.special import comb


def binomial_pmf(k, n, p):
    return comb(n, k) * (p**k) * ((1 - p) ** (n - k))

SciPy의 `comb` 함수는 `n`개 중 `k`개를 고르는 조합의 수를 계산합니다. 영어로는 흔히 “n choose k”라고 읽습니다.

`binomial_pmf`는 `p`가 주어졌을 때 `n`번 시도해 `k`번 명중할 확률을 계산합니다. 여러 `k` 값으로 함수를 호출하면 결과의 분포를 나타내는 `Pmf`를 만들 수 있습니다.

In [13]:
ks = np.arange(16, n + 1)
ps = binomial_pmf(ks, n, p)
pmf_binom = Pmf(ps, ks, name="binomial model")

시뮬레이션 결과와 비교하면 다음과 같습니다.

In [14]:
from thinkstats import two_bar_plots

two_bar_plots(pmf_sim, pmf_binom)
decorate(xlabel="Hits", ylabel="PMF")

두 분포는 비슷하고, 시뮬레이션의 우연한 변동 때문에 작은 차이가 있습니다. 시뮬레이션과 모형이 같은 가정, 특히 매 시도의 성공 확률이 같다는 가정에 바탕을 두므로 놀라운 일은 아닙니다. 모형을 더 엄격히 확인하려면 실제 자료와 비교해야 합니다.

2020년 하계 올림픽 남자 스키트 사격의 위키백과 문서에서 예선 결과 표를 가져올 수 있습니다. 자료를 내려받는 방법은 이 장의 노트북에 있습니다.

이 자료는 2024년 7월 15일 [2020년 하계 올림픽 남자 스키트 사격 문서](https://en.wikipedia.org/wiki/Shooting_at_the_2020_Summer_Olympics_–_Men%27s_skeet)에서 내려받았습니다.

In [15]:
filename = "Shooting_at_the_2020_Summer_Olympics_Mens_skeet"

In [16]:
download("https://github.com/AllenDowney/ThinkStats/raw/v3/data/" + filename)

In [17]:
tables = pd.read_html(filename)
table = tables[6]
table.head()

표에는 선수 한 명당 한 행이 있고, 다섯 라운드 각각에 해당하는 열이 있습니다. 결과가 든 열들을 선택하고 NumPy의 `flatten`으로 하나의 배열에 모으겠습니다.

In [18]:
columns = ["1", "2", "3", "4", "5"]
results = table[columns].values.flatten()

선수 30명이 다섯 라운드씩, 총 150라운드를 치렀습니다. 한 라운드에 25발씩 쏘았으므로 총 시도는 3,750번이고, 명중은 3,575번입니다.

> 번역자 주: 원문 문장에서는 시도 횟수와 명중 횟수가 뒤바뀌어 있습니다. 아래 코드의 출력에 맞춰 바로잡았습니다.

In [19]:
total_shots = 25 * len(results)
total_hits = results.sum()
n, total_shots, total_hits

따라서 전체 성공률은 95.3%입니다.

In [20]:
p = total_hits / total_shots
p

이제 `n=25`와 방금 구한 `p`를 사용해 이항분포를 나타내는 `Pmf`를 계산하겠습니다.

In [21]:
ps = binomial_pmf(ks, n, p)
pmf_binom = Pmf(ps, ks, name="binomial model")

실제 결과의 `Pmf`와 비교할 수 있습니다.

In [22]:
pmf_results = Pmf.from_seq(results, name="actual results")

two_bar_plots(pmf_results, pmf_binom)
decorate(xlabel="Hits", ylabel="PMF")

모든 선수의 실력이 똑같고 변하지 않는다는 비현실적인 가정에도 불구하고, 이항 모형은 자료의 분포에 잘 맞습니다.

## 포아송분포

운동 경기의 결과가 예측 가능한 패턴을 따르는 다른 예로, 아이스하키 경기의 득점 수를 살펴보겠습니다.

60분, 즉 3,600초짜리 경기를 시뮬레이션하는 것부터 시작합니다. 두 팀을 합쳐 한 경기에서 평균 6골을 넣는다고 하겠습니다. 매초 득점 확률 `p`가 같고, 각 초의 득점 여부는 서로 독립이라고 가정합니다.

In [23]:
n = 3600
m = 6
p = m / 3600
p

다음 함수는 `n`초를 시뮬레이션하고 총득점 수를 반환합니다.

In [24]:
def simulate_goals(n, p):
    return flip(n, p).sum()

많은 경기를 시뮬레이션하면 경기당 평균 득점 수가 6에 가까운지 확인할 수 있습니다.

In [25]:
goals = [simulate_goals(n, p) for i in range(1001)]
np.mean(goals)

이 결과를 이항분포로 설명할 수도 있지만, `n`이 크고 `p`가 작으면 **포아송분포**(Poisson distribution)로도 잘 설명됩니다. 포아송분포는 보통 그리스 문자 λ, “람다”로 나타내는 값 하나로 정해집니다. 코드에서는 `lam`을 사용합니다. `lambda`는 Python 예약어여서 변수 이름으로 쓸 수 없기 때문입니다. `lam`은 득점률을 나타내며, 이 예에서는 경기당 6골입니다.

포아송분포의 PMF도 쉽게 계산할 수 있습니다. `lam`이 주어지면 다음 함수로 한 경기에서 `k`골이 나올 확률을 계산합니다.

In [26]:
from scipy.special import factorial


def poisson_pmf(k, lam):
    """Compute the Poisson PMF.

    k (int or array-like): The number of occurrences
    lam (float): The rate parameter (λ) of the Poisson distribution

    returns: float or ndarray
    """
    return (lam**k) * np.exp(-lam) / factorial(k)

SciPy의 `factorial` 함수는 `1`부터 `k`까지의 정수를 모두 곱합니다.

여러 `k` 값으로 `poisson_pmf`를 호출하면 결과의 분포를 나타내는 `Pmf`를 만들 수 있습니다.

In [27]:
lam = 6
ks = np.arange(20)
ps = poisson_pmf(ks, lam)
pmf_poisson = Pmf(ps, ks, name="Poisson model")

분포의 평균이 6에 가까운지 확인할 수 있습니다.

In [28]:
pmf_poisson.normalize()
pmf_poisson.mean()

다음 그림은 시뮬레이션 결과와 평균이 같은 포아송분포를 비교합니다.

In [29]:
pmf_sim = Pmf.from_seq(goals, name="simulation")

two_bar_plots(pmf_sim, pmf_poisson)
decorate(xlabel="Goals", ylabel="PMF")

우연한 변동으로 생긴 작은 차이를 빼면 두 분포는 비슷합니다. 시뮬레이션과 포아송 모형 모두 경기의 매초 득점 확률이 같다고 가정하므로 놀라운 일은 아닙니다. 더 엄격히 확인하려면 실제 자료에 얼마나 잘 맞는지 보아야 합니다.

저는 HockeyReference에서 북미아이스하키리그(NHL) 2023~2024 정규 시즌의 모든 경기 결과를 내려받았습니다. 플레이오프는 포함하지 않았습니다. 연장전과 승부치기를 제외하고, 정규 경기 시간 60분 동안의 득점 정보를 추출했습니다. 결과는 경기마다 키 하나가 있는 HDF 파일에 들어 있습니다. 각 키에는 경기 시작 후 몇 초에 골이 들어갔는지 나타내는 목록이 있습니다. 자료를 내려받는 방법은 이 장의 노트북에 있습니다.

원자료는 2024년 7월 16일 [HockeyReference의 NHL 2023~2024 경기 결과](https://www.hockey-reference.com/leagues/NHL_2024_games.html)에서 내려받았습니다.

In [30]:
download("https://github.com/AllenDowney/ThinkStats/raw/v3/data/nhl_2023_2024.hdf")

파일의 키들은 다음과 같이 읽습니다.

In [31]:
filename = "nhl_2023_2024.hdf"

with pd.HDFStore(filename, "r") as store:
    keys = store.keys()

len(keys), keys[0]

정규 시즌에는 1,312경기가 있었습니다. 각 키에는 경기 날짜와 홈 팀의 세 글자 약어가 들어 있습니다. `read_hdf`로 키를 조회하면 득점 시각 목록을 얻을 수 있습니다.

In [32]:
times = pd.read_hdf(filename, key=keys[0])
times

파일에서 처음 조회한 경기에서는 여섯 골이 나왔습니다. 첫 골은 시작 424초 뒤, 마지막 골은 3,513초 뒤로, 종료까지 87초를 남긴 때였습니다.

In [33]:
3600 - times[5]

다음 반복문은 모든 경기 결과를 읽고, 경기마다 득점 수를 세어 목록에 저장합니다.

In [34]:
goals = []

for key in keys:
    times = pd.read_hdf(filename, key=key)
    n = len(times)
    goals.append(n)

경기당 평균 득점 수는 6을 조금 넘습니다.

In [35]:
lam = np.mean(goals)
lam

`poisson_pmf`로 자료와 평균이 같은 포아송분포의 `Pmf`를 만들 수 있습니다.

In [36]:
ps = poisson_pmf(ks, lam)
pmf_poisson = Pmf(ps, ks, name="Poisson model")

자료의 PMF와 비교하면 다음과 같습니다.

In [37]:
pmf_goals = Pmf.from_seq(goals, name="goals scored")

two_bar_plots(pmf_goals, pmf_poisson)
decorate(xlabel="Goals", ylabel="PMF")

포아송분포는 자료에 잘 맞습니다. 아이스하키의 득점 과정을 설명하는 좋은 모형일 수 있다는 뜻입니다.

## 지수분포

앞 절에서는 매초 득점 확률이 같은 단순한 아이스하키 경기 모형을 시뮬레이션했습니다. 이 이산 모형에서 첫 골까지의 시간은 연속적인 **지수분포**(exponential distribution)로 근사할 수 있습니다.

이를 보이려고 다시 두 팀이 평균 6골을 넣는다고 가정하고, 매초 득점할 확률을 구하겠습니다.

In [38]:
n = 3600
m = 6
p = m / 3600
p

다음 함수는 `n`초를 시뮬레이션하고 `argmax`로 첫 골의 시각을 찾습니다.

In [39]:
def simulate_first_goal(n, p):
    return flip(n, p).argmax()

`flip`의 결과는 1과 0의 시퀀스이므로 최댓값은 거의 항상 1입니다. 골이 하나라도 있으면 `argmax`는 첫 번째 골의 인덱스를 반환합니다. 골이 없으면 0을 반환하지만, 드물게 일어나므로 여기서는 무시하겠습니다.

`simulate_first_goal`로 1,001경기를 시뮬레이션하고 첫 골까지 걸린 시간을 목록으로 만들겠습니다.

In [40]:
np.random.seed(3)

In [41]:
first_goal_times = [simulate_first_goal(n, p) for i in range(1001)]
mean = np.mean(first_goal_times)
mean

첫 골까지의 평균 시간은 600초, 즉 10분에 가깝습니다. 60분짜리 경기에서 평균 6골을 기대한다면 평균 10분마다 한 골을 기대할 수 있으니 자연스러운 결과입니다.

`n`이 크고 `p`가 작을 때, 첫 골까지의 시간 분포가 지수분포에 가까워진다는 것을 수학적으로 보일 수 있습니다.

시뮬레이션에서 서로 다른 시간 값이 많이 생기므로 PMF 대신 CDF로 분포를 비교하겠습니다. 지수분포의 CDF는 쉽게 계산할 수 있습니다.

> 번역자 주: 위 시뮬레이션은 초 단위의 이산 모형이며 경기 종료 시점에서 끊깁니다. 무득점 경기의 0 반환도 포함합니다. 여기서 지수분포는 이런 단순화에 대한 연속 근사입니다.

In [42]:
def exponential_cdf(x, lam):
    """Compute the exponential CDF.

    x: float or sequence of floats
    lam: rate parameter

    returns: float or NumPy array of cumulative probability
    """
    return 1 - np.exp(-lam * x)

`lam`은 단위 시간당 평균 사건 수입니다. 여기서는 초당 득점 수입니다. 시뮬레이션 결과의 평균으로 `lam`을 구할 수 있습니다.

In [43]:
lam = 1 / mean
lam

여러 시간 값으로 함수를 호출하면 첫 골 시각의 분포를 근사할 수 있습니다. NumPy의 `linspace`는 등간격 배열을 만듭니다. 여기서는 0과 3,600을 모두 포함해 201개 값을 만듭니다.

In [44]:
from empiricaldist import Cdf

ts = np.linspace(0, 3600, 201)
ps = exponential_cdf(ts, lam)
cdf_expo = Cdf(ps, ts, name="exponential model")

다음 그림은 시뮬레이션 결과와 방금 계산한 지수분포를 비교합니다.

In [45]:
cdf_sim = Cdf.from_seq(first_goal_times, name="simulation")

cdf_expo.plot(ls=":", color="gray")
cdf_sim.plot()

decorate(xlabel="Time of first goal (seconds)", ylabel="CDF")

지수 모형은 시뮬레이션 결과에 아주 잘 맞습니다. 하지만 실제 자료에 얼마나 잘 맞는지가 더 엄격한 확인입니다.

다음 반복문은 모든 경기 결과를 읽고 첫 골의 시각을 목록에 저장합니다. 골이 없으면 `nan`을 넣습니다.

In [46]:
filename = "nhl_2023_2024.hdf"

with pd.HDFStore(filename, "r") as store:
    keys = store.keys()

In [47]:
firsts = []

for key in keys:
    times = pd.read_hdf(filename, key=key)
    if len(times) > 0:
        firsts.append(times[0])
    else:
        firsts.append(np.nan)

득점률을 추정할 때는 `nan`을 제외하고 시간의 평균을 구하는 `nanmean`을 사용할 수 있습니다.

In [48]:
lam = 1 / np.nanmean(firsts)
lam

이제 자료에서 추정한 득점률로 지수분포의 CDF를 계산할 수 있습니다.

In [49]:
ps = exponential_cdf(ts, lam)
cdf_expo = Cdf(ps, ts, name="exponential model")

자료의 CDF를 계산할 때는 `dropna=False`를 전달해 `nan` 값들을 끝에 포함하겠습니다.

In [50]:
cdf_firsts = Cdf.from_seq(firsts, name="data", dropna=False)
cdf_firsts.tail()

다음 그림은 지수분포와 자료의 분포를 비교합니다.

In [51]:
cdf_expo.plot(ls=":", color="gray")
cdf_firsts.plot()

decorate(xlabel="Time of first goal (seconds)", ylabel="CDF")

일부 구간에서는 자료가 모형에서 벗어납니다. 처음 1,000초 동안의 득점이 모형의 예측보다 적어 보입니다. 그래도 모형은 자료에 잘 맞습니다.

득점 수의 포아송 모형과 시간의 지수 모형에는 경기의 어느 초에서든 득점할 가능성이 같다는 공통 가정이 있습니다. 하키 팬에게 물어보면 아니라고 할 것이고, 그 말이 맞습니다. 현실은 이런 가정을 여러 면에서 어깁니다. 그런데도 이론분포는 실제 자료에 놀랄 만큼 잘 맞는 일이 많습니다.

## 정규분포

현실에서 측정한 값들의 분포는 **정규분포**(normal distribution)에 가까운 경우가 많습니다. 정규분포는 가우스분포라고도 하며, 그래프는 종 모양입니다. 이 분포가 어디서 나오는지 보려고 호박이 자라는 모형을 생각해 보겠습니다. 날씨가 나쁘면 하루에 1파운드, 보통이면 2파운드, 좋으면 3파운드씩 자란다고 하겠습니다. 매일 세 날씨의 확률이 같고, 날씨는 날마다 독립적으로 정해진다고 가정합니다.

다음 함수는 이 모형을 `n`일 동안 시뮬레이션하고 무게 증가량의 합을 반환합니다.

In [52]:
def simulate_growth(n):
    choices = [1, 2, 3]
    gains = np.random.choice(choices, n)
    return gains.sum()

NumPy `random` 모듈의 `choice` 함수는 값의 시퀀스에서 무작위로 `n`개를 선택한 배열을 만듭니다. 여기서는 `choices`가 그 시퀀스입니다.

서로 독립인 날씨 조건에서 호박 1,001개가 자란다고 하겠습니다. 100일의 성장 과정을 시뮬레이션하면 호박 1,001개의 무게를 얻습니다.

In [53]:
sim_weights = [simulate_growth(100) for i in range(1001)]
m, s = np.mean(sim_weights), np.std(sim_weights)
m, s

평균은 200파운드에 가깝고 표준편차는 약 8파운드입니다. 무게의 분포가 정규분포에 가까운지 다음 함수로 확인하겠습니다. 표본을 받아 그 표본과 평균·표준편차가 같은 정규분포의 `Cdf`를 만듭니다. 평균보다 4표준편차 낮은 값부터 4표준편차 높은 값까지 계산합니다.

In [54]:
from scipy.stats import norm


def make_normal_model(data):
    m, s = np.mean(data), np.std(data)
    low, high = m - 4 * s, m + 4 * s
    qs = np.linspace(low, high, 201)
    ps = norm.cdf(qs, m, s)
    return Cdf(ps, qs, name="normal model")

사용법은 다음과 같습니다.

In [55]:
cdf_model = make_normal_model(sim_weights)

이제 시뮬레이션 결과의 분포를 나타내는 `Cdf`를 만들겠습니다.

In [56]:
cdf_sim_weights = Cdf.from_seq(sim_weights, name="simulation")

다음 함수로 두 분포를 비교하겠습니다. `cdf_model`과 `cdf_data`는 `Cdf` 객체입니다. `xlabel`은 문자열이고, `options`는 `cdf_data`를 그리는 방식을 정하는 옵션 사전입니다.

In [57]:
def two_cdf_plots(cdf_model, cdf_data, xlabel="", **options):
    cdf_model.plot(ls=":", color="gray")
    cdf_data.plot(**options)
    decorate(xlabel=xlabel, ylabel="CDF")

결과는 다음과 같습니다.

In [58]:
two_cdf_plots(cdf_model, cdf_sim_weights, xlabel="Weight (pounds)")

정규 모형은 호박 무게의 분포에 잘 맞습니다. 서로 독립이고 분산이 유한한 무작위 요인들을 더하면, 일정한 조건 아래 그 합을 정규분포로 근사할 수 있습니다. [14장](https://allendowney.github.io/ThinkStats/chap14.html)에서 다시 다룰 중심극한정리의 결과입니다.

먼저 정규분포가 실제 자료에 얼마나 잘 맞는지 보겠습니다. 미국 가족성장조사(NSFG)의 출생 체중 분포를 예로 사용하겠습니다. `read_fem_preg`로 자료를 읽고, 파운드 단위 출생 체중인 `totalwgt_lb`를 선택합니다.

다음 셀들은 데이터 파일을 내려받고, 자료를 읽는 데 필요한 `statadict`를 설치합니다.

In [59]:
download("https://github.com/AllenDowney/ThinkStats/raw/v3/nb/nsfg.py")
download("https://github.com/AllenDowney/ThinkStats/raw/v3/data/2002FemPreg.dct")
download("https://github.com/AllenDowney/ThinkStats/raw/v3/data/2002FemPreg.dat.gz")

In [60]:
try:
    import statadict
except ImportError:
    %pip install statadict

In [61]:
import nsfg

preg = nsfg.read_fem_preg()
birth_weights = preg["totalwgt_lb"].dropna()

출생 체중의 평균은 약 7.27파운드이고 표준편차는 1.4파운드입니다. 하지만 앞에서 보았듯, 자료에는 오류로 보이는 이상치가 있습니다.

In [62]:
m, s = np.mean(birth_weights), np.std(birth_weights)
m, s

이상치가 평균과 표준편차의 추정에 미치는 영향을 줄이려고 SciPy의 `trimboth`로 양 끝의 값을 제외하겠습니다. `0.01`은 양쪽에서 각각 1%씩 제외한다는 뜻입니다.

In [63]:
from scipy.stats import trimboth

trimmed = trimboth(birth_weights, 0.01)
m, s = np.mean(trimmed), np.std(trimmed)
m, s

양 끝을 제외한 자료의 평균은 조금 높아지고 표준편차는 상당히 낮아집니다. 이 자료로 정규 모형을 만들겠습니다.

> 번역자 주: 원문은 평균이 낮아진다고 설명하지만, 출력은 약 7.2656에서 7.2809로 높아집니다. 출력에 맞춰 옮겼습니다.

In [64]:
cdf_model = make_normal_model(trimmed)

자료의 `Cdf`와 비교합니다.

In [65]:
cdf_birth_weight = Cdf.from_seq(birth_weights, name='data')

two_cdf_plots(cdf_model, cdf_birth_weight, xlabel="Birth weight (pounds)")

5파운드 미만을 빼면 정규 모형은 자료에 잘 맞습니다. 그 구간에서는 자료의 분포가 모형보다 왼쪽에 있습니다. 가장 가벼운 아기들이 정규분포에서 예상한 것보다 더 가볍다는 뜻입니다. 현실은 대개 간단한 수학 모형보다 복잡합니다.

## 로그정규분포

앞 절에서는 날씨에 따라 하루에 1~3파운드씩 호박이 자란다고 가정했습니다. 이번에는 성장량이 현재 무게에 비례한다고 하겠습니다. 큰 호박이 작은 호박보다 하루에 더 많이 자라는 것으로, 아마 더 현실적인 가정일 것입니다.

다음 함수는 이런 비례 성장을 시뮬레이션합니다. 날씨가 나쁘면 무게의 3%, 보통이면 5%, 좋으면 7%씩 자랍니다. 이번에도 매일 세 날씨의 확률이 같다고 가정합니다.

In [66]:
def simulate_proportionate_growth(n):
    choices = [1.03, 1.05, 1.07]
    gains = np.random.choice(choices, n)
    return gains.prod()

호박의 무게가 3% 늘면 새 무게는 원래 무게에 1.03을 곱한 값입니다. 따라서 무작위 성장 배율을 골라 모두 곱하면 100일 뒤의 무게를 계산할 수 있습니다.

이 함수를 1,001번 호출해 호박 1,001개의 무게를 저장하겠습니다.

In [67]:
sim_weights = [simulate_proportionate_growth(100) for i in range(1001)]
np.mean(sim_weights), np.std(sim_weights)

평균 무게는 약 131파운드이고 표준편차는 약 21파운드입니다. 앞 모형보다 평균은 작지만 무게의 변동은 큽니다.

이 무게의 분포는 **로그정규분포**(lognormal distribution)로 근사할 수 있습니다. 로그정규분포는 값에 로그를 취하면 정규분포가 되는 분포입니다. 이 모형에서는 날마다의 성장 배율에 로그를 취해 더한 값이 정규분포에 가까워집니다. 확인하려고 로그값의 평균과 표준편차를 구하겠습니다. 로그의 밑은 무엇이든 쓸 수 있지만, 결과를 해석하기 쉽도록 밑이 10인 로그를 사용하겠습니다.

In [68]:
log_sim_weights = np.log10(sim_weights)
m, s = np.mean(log_sim_weights), np.std(log_sim_weights)
m, s

이제 로그값의 분포와 평균·표준편차가 같은 정규분포를 비교하겠습니다.

In [69]:
cdf_model = make_normal_model(log_sim_weights)
cdf_log_sim_weights = Cdf.from_seq(log_sim_weights, name="simulation")

two_cdf_plots(
    cdf_model, cdf_log_sim_weights, xlabel="Pumpkin weight (log10 pounds)"
)

예상대로 모형은 시뮬레이션 결과에 아주 잘 맞습니다.

사람도 호박과 비슷해서 해마다 늘어나는 체중이 현재 체중에 비례한다면, 성인 체중도 로그정규분포를 따를 것으로 예상할 수 있습니다. 확인해 보겠습니다.

미국 만성질환예방·건강증진센터는 행동위험요인감시체계(BRFSS)의 일부로 해마다 조사를 실시합니다. 2008년에는 414,509명을 대상으로 인구학적 특성, 건강, 건강 위험에 관해 물었습니다. 수집한 자료에는 398,484명의 체중이 있습니다. 내려받는 방법은 이 장의 노트북에 있습니다.

In [70]:
download("https://github.com/AllenDowney/ThinkStats/raw/v3/data/CDBRFS08.ASC.gz")

`thinkstats` 모듈에는 BRFSS 자료를 읽고 pandas `DataFrame`을 반환하는 함수가 있습니다.

In [71]:
from thinkstats import read_brfss

brfss = read_brfss()

성인의 킬로그램 단위 체중은 `wtkg2` 열에 기록되어 있습니다.

In [72]:
adult_weights = brfss["wtkg2"].dropna()
m, s = np.mean(adult_weights), np.std(adult_weights)
m, s

평균은 약 79kg입니다. 로그를 구하기 전에 체중이 정규분포를 따르는지 먼저 보겠습니다.

In [73]:
cdf_model = make_normal_model(adult_weights)
cdf_adult_weights = Cdf.from_seq(adult_weights, name="adult weight")

two_cdf_plots(cdf_model, cdf_adult_weights, xlabel="Adult weight (kilograms)")

목적에 따라서는 정규분포가 이 자료를 충분히 잘 설명할 수 있습니다. 하지만 더 나은 모형이 있는지 보겠습니다.

다음은 체중의 로그값 분포와 평균·표준편차가 같은 정규 모형입니다.

In [74]:
log_adult_weights = np.log10(adult_weights)
cdf_model = make_normal_model(log_adult_weights)

cdf_log_adult_weights = Cdf.from_seq(log_adult_weights, name="log adult weight")

In [75]:
two_cdf_plots(cdf_model, cdf_log_adult_weights, xlabel="Adult weight (log10 kg)")

정규 모형은 체중 자체보다 로그값에 더 잘 맞습니다. 이는 비례 성장 모형과 부합하는 결과이지만, 분포의 모양만으로 실제 성장 과정을 확정할 수는 없습니다.

## 왜 모형을 만들까?

이 장의 시작에서 현실의 많은 현상을 이론분포로 설명할 수 있다고 했습니다. 하지만 왜 여기에 관심을 가져야 하는지는 분명하지 않았을 수 있습니다.

모든 모형이 그렇듯 이론분포도 추상화입니다. 관련이 없다고 여긴 세부 사항을 생략한다는 뜻입니다. 예를 들어 관측한 분포에는 측정 오류나 특정 표본만의 특이한 점이 있을 수 있습니다. 이론 모형은 그런 개별적인 특성을 무시합니다.

이론 모형은 데이터 압축의 한 형태이기도 합니다. 모형이 자료에 잘 맞으면 적은 수의 숫자로 많은 양의 자료를 요약할 수 있습니다.

자연 현상의 자료가 이론분포에 맞는다는 것이 놀라울 때도 있지만, 이런 관측은 실제 체계를 이해하는 단서가 될 수 있습니다. 관측된 분포가 왜 특정한 형태인지 설명할 수도 있습니다. 예를 들어 앞 절에서 성인 체중이 로그정규분포로 잘 설명된다는 것을 보았습니다. 체중 변화가 현재 체중에 비례한다는 모형을 검토할 단서가 됩니다.

또한 이론분포는 [14장](https://allendowney.github.io/ThinkStats/chap14.html)에서 보겠듯 수학적으로 분석하기 좋습니다.

하지만 모든 모형은 불완전하다는 점을 기억해야 합니다. 현실의 자료는 이론분포에 완벽하게 들어맞지 않습니다. 사람들은 때때로 자료가 모형에서 만들어진 것처럼 말합니다. 사람의 키는 정규분포이고 소득은 로그정규분포라고 하는 식입니다. 이 말을 문자 그대로 받아들이면 참일 수 없습니다. 현실과 수학 모형 사이에는 언제나 차이가 있습니다.

모형은 현실의 관련 있는 면을 담고 필요 없는 세부 사항을 생략할 때 유용합니다. 무엇이 관련 있고 무엇이 불필요한지는 모형을 어디에 쓰려는지에 따라 달라집니다.

## 용어 정리

- **이항분포**(binomial distribution): 성공 확률이 같은 독립 시행을 정해진 횟수만큼 반복할 때 성공 횟수의 분포.
- **포아송분포**(Poisson distribution): 일정 시간 동안 일어나는 사건 수를 설명할 때 자주 쓰는 이론분포.
- **지수분포**(exponential distribution): 사건 사이의 시간을 설명할 때 자주 쓰는 이론분포.
- **정규분포**(normal distribution): 대칭적인 종 모양 곡선을 따르는 자료를 설명할 때 자주 쓰는 이론분포.
- **로그정규분포**(lognormal distribution): 양수인 값에 로그를 취하면 정규분포가 되는 분포.

## 연습문제

### 연습문제 5.1

NSFG 응답자 파일의 `numfmhh` 열에는 각 응답자의 가구에 사는 가족 구성원 수가 기록되어 있습니다. `read_fem_resp`로 파일을 읽고, `query`로 면접 당시 25세 이상이었던 응답자를 선택할 수 있습니다.

In [76]:
download("https://github.com/AllenDowney/ThinkStats/raw/v3/data/2002FemResp.dct")
download("https://github.com/AllenDowney/ThinkStats/raw/v3/data/2002FemResp.dat.gz")

In [77]:
from nsfg import read_fem_resp

resp = read_fem_resp()

In [78]:
older = resp.query("age >= 25")
num_family = older["numfmhh"]

25세 이상 응답자의 `numfmhh`로 `Pmf`를 만들고, 평균이 같은 포아송분포와 비교하세요. 포아송 모형은 자료에 얼마나 잘 맞나요?

In [79]:
# Solution goes here

In [80]:
# Solution goes here

In [81]:
# Solution goes here

In [82]:
# Solution goes here

In [83]:
# Solution goes here

In [84]:
# Solution goes here

### 연습문제 5.2

이 장 앞부분에서 하키의 첫 골까지 걸리는 시간이 지수분포를 따르는 것을 보았습니다. 득점 모형이 맞다면 이전 골 뒤로 얼마나 지났는지와 상관없이 어느 때나 득점할 가능성이 같습니다. 그렇다면 골과 골 사이의 시간도 지수분포를 따를 것으로 예상할 수 있습니다.

다음 반복문은 하키 자료를 다시 읽고, 두 골 이상 나온 경기에서 연속 득점 사이의 시간을 구해 목록에 모읍니다.

In [85]:
filename = "nhl_2023_2024.hdf"

with pd.HDFStore(filename, "r") as store:
    keys = store.keys()

In [86]:
intervals = []

for key in keys:
    times = pd.read_hdf(filename, key=key)
    if len(times) > 1:
        intervals.extend(times.diff().dropna())

관측한 득점 간격과 평균이 같은 지수분포의 CDF를 `exponential_cdf`로 계산하세요. 모형과 자료의 CDF를 비교하세요.

In [87]:
# Solution goes here

In [88]:
# Solution goes here

In [89]:
# Solution goes here

In [90]:
# Solution goes here

In [91]:
# Solution goes here

### 연습문제 5.3

사람의 키 분포는 정규분포와 로그정규분포 중 어디에 더 가까울까요? 알아보기 위해 다음과 같이 BRFSS에서 키를 선택할 수 있습니다.

In [92]:
adult_heights = brfss["htm3"].dropna()
m, s = np.mean(adult_heights), np.std(adult_heights)
m, s

이 값들의 CDF를 구하고, 평균과 표준편차가 같은 정규분포와 비교하세요. 이어서 키에 로그를 취하고, 로그값의 분포를 정규분포와 비교하세요. 눈으로 비교했을 때 어느 모형이 자료에 더 잘 맞나요?

In [93]:
# Solution goes here

In [94]:
# Solution goes here

In [95]:
# Solution goes here

In [96]:
# Solution goes here

In [97]:
# Solution goes here

In [98]:
# Solution goes here

[Think Stats: Exploratory Data Analysis in Python, 3rd Edition](https://allendowney.github.io/ThinkStats/index.html)

Copyright 2024 [Allen B. Downey](https://allendowney.com)

코드 라이선스: [MIT](https://mit-license.org/)

본문·그림과 한국어 번역 라이선스: [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)